In [ ]:
import pandas as pd


In [ ]:
fp = "../data/olist_prepared/sba_loans_stage_1.csv"
df = pd.read_csv(fp)

In [ ]:
df

In [ ]:
cardinality = {}
att_list = ["BorrCity", "BorrZip", "BankCity", "BankZip", "BorrState", "BankState"]
for a in att_list:
    num_vals = len(df[a].unique())
    cardinality[a] = num_vals

In [ ]:
cardinality_df = pd.DataFrame.from_dict(cardinality, orient="index").reset_index()
cardinality_df.columns = ["Attribute", "Cardinality"]

In [ ]:
cardinality_df

In [ ]:
df["BorrState"].value_counts()

In [ ]:
cols_needed = ["BorrState", "LoanStatus"]
dfbs = df[cols_needed]

In [ ]:
dfbs.LoanStatus.value_counts()

In [ ]:
dfbs.loc[:, "LoanStatus"] = dfbs.loc[:, "LoanStatus"].replace({"PIF":0, "CHGOFF": 1})

In [ ]:
types_bs = {"BorrState": 'category', "LoanStatus": 'float'}
dfbs = dfbs.astype(types_bc)

In [ ]:
state_list_series = dfbc["BorrState"]

In [ ]:
dfbs.dtypes

In [ ]:
dfbs = pd.get_dummies(dfbs, drop_first=True, dtype=float)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
col_pred = dfbs.columns.tolist()
col_pred.remove("LoanStatus")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(dfbs[col_pred], dfbs["LoanStatus"], test_size=0.2, random_state=42)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
import cvxpy as cp
import numpy as np

In [ ]:
num_features = X_train.shape[1]

In [ ]:
X_train.shape[1]

In [ ]:
def error(scores, labels):
  scores[scores > 0] = 1
  scores[scores <= 0] = 0
  return np.sum(np.abs(scores - labels)) / float(np.size(labels))

In [ ]:
# Define variables
beta = cp.Variable(num_features)

# Define the negative log-likelihood loss for logistic regression
loss = cp.sum(cp.logistic(X_train.values  @ beta) - cp.multiply(y_train.values, X_train.values @ beta))
ll_vals = np.logspace(1,8,4, base=2)
llf_vals = np.logspace(1,8,4, base=2)
train_error_vals = []
test_error_vals = []

for llv in ll_vals:
    
    for llf in llf_vals:
        
        # Define Lasso penalty
        lambda_lasso = cp.Parameter(nonneg=True) # Regularization parameter for Lasso
        lasso_penalty = lambda_lasso * cp.norm1(beta)

        # Define Fused Lasso penalty (if applicable, for ordered features)
        lambda_fused = cp.Parameter(nonneg=True) # Regularization parameter for Fused Lasso
        fused_penalty = lambda_fused * cp.norm1(beta[1:] - beta[:-1]) # Example for 1D fusion

        # Combine objective
        objective = cp.Minimize(loss + lasso_penalty + fused_penalty) # Or just loss + lasso_penalty
    
        # Create and solve the problem
        problem = cp.Problem(objective)

        # Set lambda_lasso and lambda_fused (if used) and solve
        lambda_lasso.value = llv
        lambda_fused.value = llf
        problem.solve(solver=cp.CLARABEL)
        train_error = error( (X_train.values @ beta).value, y_train.values)
        test_error = error( (X_test.values @ beta).value, y_test.values)
        train_error_vals.append((llv.item(), llf.item(), round(train_error.item(), 4)))
        test_error_vals.append((llv.item(), llf.item(), round(test_error.item(),4)))
    
    

    

In [ ]:
test_error_vals = sorted(test_error_vals, key=lambda x: x[2])

In [ ]:
train_error_vals = sorted(train_error_vals, key=lambda x: x[2])

In [ ]:
test_error_vals

In [ ]:
ll_vals = np.linspace(0,.1,4)
llf_vals = np.linspace(0,.05,4)
train_error_vals = []
test_error_vals = []

for llv in ll_vals:
    
    for llf in llf_vals:
        
        # Define Lasso penalty
        lambda_lasso = cp.Parameter(nonneg=True) # Regularization parameter for Lasso
        lasso_penalty = lambda_lasso * cp.norm1(beta)

        # Define Fused Lasso penalty (if applicable, for ordered features)
        lambda_fused = cp.Parameter(nonneg=True) # Regularization parameter for Fused Lasso
        fused_penalty = lambda_fused * cp.norm1(beta[1:] - beta[:-1]) # Example for 1D fusion

        # Combine objective
        objective = cp.Minimize(loss + lasso_penalty + fused_penalty) # Or just loss + lasso_penalty
    
        # Create and solve the problem
        problem = cp.Problem(objective)

        # Set lambda_lasso and lambda_fused (if used) and solve
        lambda_lasso.value = llv
        lambda_fused.value = llf
        problem.solve(solver=cp.CLARABEL)
        train_error = error( (X_train.values @ beta).value, y_train.values)
        test_error = error( (X_test.values @ beta).value, y_test.values)
        train_error_vals.append((llv.item(), llf.item(), round(train_error.item(), 4)))
        test_error_vals.append((llv.item(), llf.item(), round(test_error.item(),4)))

In [ ]:
test_error_vals = sorted(test_error_vals, key=lambda x: x[2])
test_error_vals

In [ ]:
pd.DataFrame(beta.value).hist()

In [ ]:
df_beta_borr_state = pd.DataFrame(beta.value)

In [ ]:
dsl = dfbs.columns.tolist()

In [ ]:
default_state = sorted(state_list_series)[0]
if "LoanStatus" in dsl:
    dsl.remove("LoanStatus")


In [ ]:
df_beta_borr_state.loc[:, "BorrState"] = dsl

In [ ]:
df_beta_borr_state.columns = ["Coeff", "BorrState"]

In [ ]:
col_order = ["BorrState", "Coeff"]
df_beta_borr_state = df_beta_borr_state[col_order]

In [ ]:
df_beta_borr_state = df_beta_borr_state.sort_values(by="Coeff")

In [ ]:
df_beta_borr_state

In [ ]:
df_beta_borr_state.Coeff.diff()